Importing packages

In [1]:
import cv2
import os
import random
import shutil
import matplotlib.pyplot as plt

In [2]:
def display_image_with_bounding_boxes(image_path, bounding_box_path):
    # Paths
    IMAGE_PATH = image_path
    BOUNDING_BOX_PATH = bounding_box_path

    # Load the image
    image = cv2.imread(IMAGE_PATH)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    img_height, img_width, _ = image.shape

    # Load the labels
    with open(BOUNDING_BOX_PATH, 'r') as f:
        lines = f.readlines()

    # Draw bounding boxes
    for line in lines:
        parts = line.strip().split()
        class_id = int(parts[0])
        x_center = float(parts[1])
        y_center = float(parts[2])
        width = float(parts[3])
        height = float(parts[4])

        # Convert normalized coordinates to pixel coordinates
        x_center *= img_width
        y_center *= img_height
        width *= img_width
        height *= img_height

        # Calculate top-left and bottom-right corners
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)

        # Clamp coordinates to image boundaries
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_width - 1, x2)
        y2 = min(img_height - 1, y2)

        # Draw rectangle
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Put class label
        cv2.putText(
            image,
            f"Class {class_id}",
            (x1, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
        )   
    h, w, _ = image.shape

    plt.figure(figsize=(w/100, h/100), dpi=100)
    plt.imshow(image)
    plt.axis("off")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.show()

In [3]:
def create_YOLO_ready_data_distribution(root):
    ROOT_DIR = root
    IMAGES_DIR = os.path.join(ROOT_DIR, 'images')
    LABELS_DIR = os.path.join(ROOT_DIR, 'labels')

    # Create proper YOLO folders
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(ROOT_DIR, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(ROOT_DIR, 'labels', split), exist_ok=True)

    # Get all images
    images = [f for f in os.listdir(IMAGES_DIR) if f.endswith(('.jpg', '.png'))]

    # Shuffle
    random.seed(1)
    random.shuffle(images)

    # Split 80% train, 20% val
    test_index = int(0.9 * len(images))
    train_images = images[:int(0.8*len(images))]
    val_images = images[int(0.8*len(images)):test_index]
    test_images = images[test_index:]

    def move_files(file_list, split):
        for img_file in file_list:
            label_file = img_file.rsplit(".", 1)[0] + ".txt"

            # Move image
            shutil.move(
                os.path.join(IMAGES_DIR, img_file),
                os.path.join(ROOT_DIR, 'images', split, img_file)
            )

            # Move corresponding label
            shutil.move(
                os.path.join(LABELS_DIR, label_file),
                os.path.join(ROOT_DIR, 'labels', split, label_file)
            )

    move_files(train_images, 'train')
    move_files(val_images, 'val')
    move_files(test_images, 'test')

    print(f"Moved {len(train_images)} train, {len(val_images)} val, {len(test_images)} test images.")

In [4]:
display_image_with_bounding_boxes('DeepSpaceYoloDataSet/images/5.jpg', 'DeepSpaceYoloDataSet/labels/5.txt')

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [5]:
create_YOLO_ready_data_distribution('DeepSpaceYoloDataSet')

Moved 0 train, 0 val, 0 test images.


In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3070 Ti


In [9]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="DeepSpaceYoloDataset/data.yaml",
    imgsz=608,
    batch=16,
    device=0
)

Ultralytics 8.4.19  Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=DeepSpaceYoloDataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=608, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000012F3DAF5700>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480